In [ ]:
#import libraries / load customs data set
import pandas as pd

df = pd.read_csv("bettergov.ph 2015.csv", encoding="cp1252")

print(df.shape)

print(df.columns.tolist())
df.head()


C:\Users\Lance Santos\AppData\Local\Temp\ipykernel_4704\846233825.py:3: DtypeWarning: Columns (4: entry, 25: prefcode, 28: subport, 29: port) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("bettergov.ph 2015.csv", encoding="cp1252")


(2236612, 30)
['uid', 'ty', 'tq', 'tm', 'entry', 'hscode', 'goodsdescription', 'p', 'q', 'm_fob', 'm_cif', 'fx_usd', 'dutiablevalueforeign', 'exchangerate', 'currency', 'dutiablevaluephp', 'dutypaid', 'exciseadvalorem', 'arrastre', 'wharfage', 'vatbase', 'vatpaid', 'othertax', 'finesandpenalties', 'dutiestaxes', 'prefcode', 'countryorigin_iso3', 'countryexport_iso3', 'subport', 'port']


,uid,ty,tq,tm,entry,hscode,goodsdescription,p,q,m_fob,...,vatbase,vatpaid,othertax,finesandpenalties,dutiestaxes,prefcode,countryorigin_iso3,countryexport_iso3,subport,port
0,201501 00000001,2015,2015q1,2015m1,C,15119090000,RBD PALM OLEIN IN BULK,0.643583,2999325.0,1930315.60,...,90638835,10876660,NaN,NaN,10876660,AFTA,MYS,MYS,Sub-Port of Dumaguete,Port of Cebu
1,201501 00000002,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.632000,532480.0,336527.38,...,15884083,1906090,NaN,NaN,1906090,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
2,201501 00000003,2015,2015q1,2015m1,C,72104990000,PRIME HOT DIPPED GALVANIZED STEEL SHEET IN C,0.627000,1779450.0,1115715.50,...,53064898,6367787,NaN,NaN,6367787,ACFTA,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
3,201501 00000004,2015,2015q1,2015m1,C,72139100000,HOT ROLLED WIRE ROD SWRY11 6.5MM,0.474000,1525249.0,722968.00,...,34664620,4159754,NaN,NaN,4501570,NaN,CHN,CHN,Harbour Centre Port Terminal Inc,Port of Manila
4,201501 00000005,2015,2015q1,2015m1,C,72163300000,STEEL STRUCTURE,1.202257,134305.2,161469.44,...,7555345,906641,NaN,NaN,906641,AKFTA,KOR,KOR,Harbour Centre Port Terminal Inc,Port of Manila


In [ ]:
#inspect dutiablevaluephp
print(df["dutiablevaluephp"].dtype)
print(df["dutiablevaluephp"].describe())
print(df["dutiablevaluephp"].isna().sum())

int64
count    2.236612e+06
mean     1.603885e+06
std      2.479364e+07
min      1.000000e+00
25%      1.901500e+04
50%      1.016945e+05
75%      5.849100e+05
max      8.367496e+09
Name: dutiablevaluephp, dtype: float64
0


In [3]:
#convert to a NumPy array
import numpy as np

values = df["dutiablevaluephp"].dropna().to_numpy()

print(type(values))
print(values.shape)

<class 'numpy.ndarray'>
(2236612,)


In [4]:
#boolean mask + aggregation
threshold = 1000000

mask = values > threshold

filtered_values = values[mask]

print("Number above threshold:", len(filtered_values))
print("Total:", np.sum(filtered_values))



Number above threshold: 411305
Total: 3288821943420


In [ ]:
#fixed seed sample
rng = np.random.default_rng(42)

sample = rng.choice(values, size=10000, replace=False)
print(sample.shape)

(10000,)


In [6]:
#vectorized caluclation / calculates 10% of each value in the sample at once using numpy

vectorized_result = sample* 0.10 
print(vectorized_result[:5])

[10328.6 28277.1  5562.4  4929.4  3778. ]


In [7]:
#same calculation with a loop

loop_result = []
for value in sample:
    loop_result.append(value * 0.10)

loop_result = np.array(loop_result)

print(loop_result[:5])

[10328.6 28277.1  5562.4  4929.4  3778. ]


In [ ]:
#verify equal results
print(np.allclose(vectorized_result, loop_result))

True


In [ ]:
#benchmark 5 runs
import time

loop_times = []
vectorized_times = []

for _ in range(5):
    # Time loop calculation
    start = time.perf_counter()

    result = []
    for value in sample:
        result.append(value * 0.10)

    result = np.array(result)

    end = time.perf_counter()
    loop_times.append(end - start)

    # Time vectorized calculation
    start = time.perf_counter()

    result = sample * 0.10

    end = time.perf_counter()
    vectorized_times.append(end - start)

print("Loop times:", loop_times)
print("Vectorized times:", vectorized_times)

print("Loop median:", np.median(loop_times))
print("Vectorized median:", np.median(vectorized_times))

Loop times: [0.017693000030703843, 0.020277799980249256, 0.016187000030186027, 0.01622510002925992, 0.015353000024333596]
Vectorized times: [6.930000381544232e-05, 3.4299970138818026e-05, 2.4799956008791924e-05, 2.4800014216452837e-05, 0.00012440001592040062]
Loop median: 0.01622510002925992
Vectorized median: 3.4299970138818026e-05


In [ ]:
#median timing
np.median(loop_times)
np.median(vectorized_times)

np.float64(3.4299970138818026e-05)

In [11]:
from src.numpy_ops import (
    apply_mask,
    vectorized_calculation,
    loop_calculation,
    aggregate_values
)

In [12]:
filtered_values = apply_mask(values, 1_000_000)

print("Number above threshold:", len(filtered_values))
print("Total:", aggregate_values(filtered_values))

Number above threshold: 411305
Total: 3288821943420.0


In [13]:
vectorized_result = vectorized_calculation(sample)
loop_result = loop_calculation(sample)

print(np.allclose(vectorized_result, loop_result))

True


In [14]:
#testing
from src.plots import make_bar_plot, make_heatmap

print("Plot functions imported successfully!")

Plot functions imported successfully!


In [15]:

#create a test data frame
test_data = pd.DataFrame({
    "category": ["A", "B", "C", "D"],
    "value": [100, 250, 175, 300]
})

In [16]:
#test reusable bar function
make_bar_plot(
    data=test_data,
    x_column="category",
    y_column="value",
    title="Test Bar Plot",
    x_label="Category",
    y_label="Value",
    output_path="test_bar.png",
)

In [ ]:
#test heatmap
heatmap_data = np.array([
    [10, 20, 30],
    [25, 15, 35],
    [40, 30, 20]
])




In [19]:
make_heatmap(
    data=heatmap_data,
    title="Test Heatmap",
    x_label="Column",
    y_label="Row",
    output_path="test_heatmap.png",
)